# library

In [81]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics.pairwise import cosine_similarity

# load data

In [82]:
customer_profile = pd.read_csv(
    r"D:\Sem 4\Rekomendasi Sistem\Tugas\PA\data\processed_customer_segmented.csv"
)

rules = pd.read_csv(
    r"D:\Sem 4\Rekomendasi Sistem\Tugas\PA\data\processed_association_rules.csv"
)

similarity_matrix = joblib.load(
    r"D:\Sem 4\Rekomendasi Sistem\Tugas\PA\src\customer_similarity.pkl"
)

# Function Similar Customer

In [83]:
def get_similar_customers(customer_id, top_n=5):

    idx = customer_profile[
        customer_profile['CustomerID'] == customer_id
    ].index[0]

    similarity_scores = list(
        enumerate(similarity_matrix[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    similar_indices = [
        i[0]
        for i in similarity_scores
    ]

    return customer_profile.iloc[
        similar_indices
    ]

# Function Promo Recommendation

In [84]:
def get_promotion(segment):

    if segment == "Loyal Customer":

        return [
            "Cashback 20%",
            "Voucher Rp50.000"
        ]

    elif segment == "Potential Customer":

        return [
            "Diskon 15%",
            "Voucher Rp25.000"
        ]

    elif segment == "At Risk Customer":

        return [
            "Voucher Comeback Rp100.000",
            "Free Shipping"
        ]

    else:

        return [
            "Welcome Voucher",
            "Diskon Pembelian Pertama"
        ]

# Function Bundling Recommendation

In [85]:
rules['antecedents'] = rules[
    'antecedents'
].astype(str)

rules['consequents'] = rules[
    'consequents'
].astype(str)

In [86]:
import re

def get_bundle(category):

    bundle = rules[
        rules['antecedents']
        .str.contains(category, regex=False)
    ]

    bundle = bundle.sort_values(
        by='confidence',
        ascending=False
    )

    recommendations = []

    for item in bundle['consequents']:

        # ambil isi dalam tanda kutip
        cats = re.findall(
            r"'([^']*)'",
            item
        )

        for cat in cats:

            if cat != category:

                recommendations.append(
                    f"{category} + {cat}"
                )

    recommendations = list(
        dict.fromkeys(recommendations)
    )

    return recommendations[:3]

# hybrid fungsi

In [87]:
def hybrid_recommendation(customer_id):

    customer = customer_profile[
        customer_profile['CustomerID']
        == customer_id
    ]

    if len(customer) == 0:

        return "Customer tidak ditemukan"

    segment = customer[
        'Segment'
    ].values[0]

    favorite_category = customer[
        'FavoriteCategory'
    ].values[0]

    similar_customers = get_similar_customers(
        customer_id
    )

    promo = get_promotion(
        segment
    )

    bundling = get_bundle(
        favorite_category
    )

    return {

        "CustomerID":
        customer_id,

        "Segment":
        segment,

        "FavoriteCategory":
        favorite_category,

        "Promo":
        promo,

        "Bundling":
        bundling,

        "Similar_Customers":
        similar_customers[
            'CustomerID'
        ].tolist()

    }

# testing

In [88]:
result = hybrid_recommendation(
    "CUST00001"
)

result

{'CustomerID': 'CUST00001',
 'Segment': 'Potential Customer',
 'FavoriteCategory': 'Electronics',
 'Promo': ['Diskon 15%', 'Voucher Rp25.000'],
 'Bundling': ['Electronics + Home',
  'Electronics + Beauty',
  'Electronics + Sport'],
 'Similar_Customers': ['CUST00449',
  'CUST01625',
  'CUST01632',
  'CUST01926',
  'CUST00068']}

In [89]:
customer_id = "CUST00022"

result = hybrid_recommendation(
    customer_id
)

print("="*50)

print(
    "CUSTOMER :",
    result['CustomerID']
)

print(
    "SEGMENT :",
    result['Segment']
)

print(
    "FAVORITE CATEGORY :",
    result['FavoriteCategory']
)

print("\nPROMO RECOMMENDATION")

for promo in result['Promo']:

    print("-", promo)

print("\nBUNDLING RECOMMENDATION")

for item in result['Bundling']:

    print("-", item)

print("\nSIMILAR CUSTOMERS")

for cust in result[
    'Similar_Customers'
]:

    print("-", cust)

CUSTOMER : CUST00022
SEGMENT : New Customer
FAVORITE CATEGORY : Fashion

PROMO RECOMMENDATION
- Welcome Voucher
- Diskon Pembelian Pertama

BUNDLING RECOMMENDATION
- Fashion + Home
- Fashion + Electronics
- Fashion + Beauty

SIMILAR CUSTOMERS
- CUST00344
- CUST01908
- CUST00276
- CUST00391
- CUST00549


In [90]:
print(type(rules['antecedents'].iloc[0]))
print(type(rules['consequents'].iloc[0]))

rules[['antecedents','consequents']].head()

<class 'str'>
<class 'str'>


,antecedents,consequents
0,frozenset({'Beauty'}),frozenset({'Electronics'})
1,frozenset({'Electronics'}),frozenset({'Beauty'})
2,frozenset({'Fashion'}),frozenset({'Beauty'})
3,frozenset({'Beauty'}),frozenset({'Fashion'})
4,frozenset({'Beauty'}),frozenset({'Home'})
